# Evaluation

Evaluate any model on `tiny_val.txt` and log results to `training_log.csv`.

Set `SOURCE` and `SOURCE_TYPE` in the config cell, then run all cells.

| source_type | source example |
|---|---|
| `"custom"` | `"checkpoints/my-run.pt"` |
| `"hf"` | `"gpt2"` or `"EleutherAI/pythia-70m"` |
| `"fft"` | `"checkpoints/my-fft/"` |
| `"lora"` | `"checkpoints/my-lora/"` |

In [ ]:
SOURCE      = "gpt2"
SOURCE_TYPE = "hf"       # "custom" | "hf" | "fft" | "lora"
DEVICE      = "cuda"
VAL_PATH    = "../data/tiny_val.txt"
NOTE        = ""

In [ ]:
from load import load_model

model, tokenizer = load_model(SOURCE, SOURCE_TYPE, device=DEVICE)
print(f"Loaded {SOURCE_TYPE}: {SOURCE}")

In [ ]:
from eval import evaluate

# For custom models, also pass cfg (loaded alongside the model)
cfg = model.cfg if SOURCE_TYPE == "custom" else None

loss = evaluate(model, VAL_PATH, tokenizer, SOURCE_TYPE, cfg=cfg, device=DEVICE)
print(f"val loss: {loss:.4f}")

In [ ]:
import math
import types
from utils import log_results

with open(VAL_PATH) as f:
    val_text = f.read()

if SOURCE_TYPE == "custom":
    ids = tokenizer.encode(val_text).ids
else:
    ids = tokenizer.encode(val_text)

avg_chars_per_token = len(val_text) / len(ids)
bpc = loss / math.log(2) / avg_chars_per_token
print(f"BPC: {bpc:.4f}  (avg chars/token: {avg_chars_per_token:.3f})")

# log_results expects a cfg-like object; use a namespace for HF models
log_cfg = cfg if SOURCE_TYPE == "custom" else types.SimpleNamespace(
    vocab_size=None, layers=None, d_model=None, d_ff=None, H=None,
    B=None, n=None, lr=None, epochs=None
)

log_results(
    cfg=log_cfg,
    train_loss=0.0,
    val_loss=loss,
    tiny_val_loss=loss,
    avg_chars_per_token=avg_chars_per_token,
    name=SOURCE,
    note=NOTE,
    csv_path="../training_log.csv",
)